# 17 — ORB Feature Detection and Matching

**Section:** Perception · **Mirrors MATLAB:** *Feature Detection, Extraction, and Matching*

ORB (Oriented FAST and Rotated BRIEF) is a fast, rotation-invariant feature detector + descriptor. We detect features in two synthetic images (one is a rotated + translated version of the other) and match them with Hamming distance + cross-check.

## Intuition — what's actually going on?

When you see the same object from two angles, your brain effortlessly recognizes it's the same thing. A camera can't do that directly — but it can be taught to spot *distinctive points* (corners, blobs) and produce a fingerprint (descriptor) for each one. Two images of the same scene then share many fingerprints, and we can match them up.

ORB has two parts:

- **Detector** (FAST): scan every pixel. Look at the 16 pixels in a small circle around it. If at least 9 contiguous pixels are all significantly brighter or all significantly darker, it's a corner. Fast because it's mostly comparisons.
- **Descriptor** (rotated BRIEF): around each corner, pick ~256 pre-chosen pairs of nearby pixels. For each pair, write a 1 if the first is darker than the second, else 0. That's a 256-bit fingerprint. "Rotated" means the pairs are pre-rotated by the corner's orientation, so the fingerprint doesn't change if you turn the image.

Matching is then bit-counting: two fingerprints' similarity is just how many bits they share (Hamming distance). Extremely fast on CPUs.

ORB is the workhorse of mobile-robot visual odometry and SLAM — see notebook 11 (ICP) for how matched features turn into a rigid-transform estimate.

### Analytical underpinning + compatibility

**FAST corner test.** A pixel $p$ with intensity $I_p$ is a corner if, among the 16 pixels on a circle of radius 3 around it, at least $N=9$ contiguous pixels satisfy $I_i > I_p + t$ (brighter test) OR at least 9 satisfy $I_i < I_p - t$ (darker test), where $t$ is a fixed intensity threshold (typically $t = 0.2 \cdot 255 = 51$ on 8-bit images). The 9-of-16 contiguous-arc requirement makes the test fast and rotationally meaningful.

**Orientation assignment.** Compute the intensity centroid of a patch around the corner; the vector from corner center to centroid defines the orientation angle $	heta$.

**BRIEF descriptor.** Pre-select $n$ (typically 256) pairs of pixel locations $(p_i, q_i)$ within a patch around the corner. The descriptor is the binary string

$$d_i = egin{cases} 1 & 	ext{if } I(p_i) < I(q_i) \ 0 & 	ext{otherwise} \end{cases},\quad i = 1, \ldots, n$$

For ORB, the pixel pairs are rotated by the orientation $	heta$ before sampling — that's the "Rotated" in "Rotated BRIEF" and is what makes ORB rotation-invariant.

**Matching.** Distance between two BRIEF descriptors is the **Hamming distance** = number of bit positions where they differ = `popcount(d_1 XOR d_2)`. Fast brute-force pairwise comparison is feasible because descriptors are binary.

**Significance test for matches.** Two *random* 256-bit descriptors have expected Hamming distance $n/2 = 128$ with standard deviation $\sqrt{n}/2 = 8$ (council fix, Wald). A true match should have distance well below this — production threshold typically $< n/3 pprox 85$ to reject false matches; distances below $\sim 64$ are essentially certain matches. The 0-bit zero distance you might naively use as the only "match" criterion is far too strict for real images.

**Cross-check.** Keep only pairs where $d_a$ is the closest match for $d_b$ *and* vice versa. Greatly reduces false matches. In visual-SLAM pipelines, cross-checked matches are further filtered by **RANSAC** on the rigid transform (or essential matrix for stereo): keep only the largest set of matches consistent with one geometric transform.

### Compatibility check — math ↔ code

| Math | Code |
|---|---|
| FAST corner + rotated BRIEF, up to 300 features | `orb = cv2.ORB_create(nfeatures=300)` |
| Detect keypoints + descriptors | `kp, des = orb.detectAndCompute(img, None)` (descriptor is 32-byte binary) |
| Hamming-distance matcher with cross-check | `bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)` |
| Match + sort by descriptor distance | `matches = sorted(bf.match(des1, des2), key=lambda m: m.distance)[:40]` |


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

H, W = 240, 320
img1 = np.zeros((H, W), dtype=np.uint8)
cv2.rectangle(img1, (50, 50), (100, 100), 200, -1)
cv2.circle(img1, (200, 80), 30, 150, -1)
cv2.line(img1, (60, 150), (260, 200), 250, 3)
cv2.putText(img1, 'PYROBOT', (30, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.0, 255, 2)
img1 = cv2.GaussianBlur(img1, (3, 3), 0)

M = cv2.getRotationMatrix2D((W / 2, H / 2), 20, 1.0)
M[0, 2] += 25
M[1, 2] += 10
img2 = cv2.warpAffine(img1, M, (W, H))


In [ ]:
orb = cv2.ORB_create(nfeatures=300)
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = sorted(bf.match(des1, des2), key=lambda m: m.distance)[:40]
print(f"img1 features: {len(kp1)}  img2 features: {len(kp2)}  matches kept: {len(matches)}")

out = cv2.drawMatches(img1, kp1, img2, kp2, matches, None,
                       flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)


In [ ]:
plt.figure(figsize=(14, 6))
plt.imshow(out, cmap='gray')
plt.title('ORB Feature Matches (img1 ↔ rotated/translated img2)')
plt.axis('off')
plt.tight_layout()
plt.show()


## References & rigor notes

**Rotation invariance.** FAST itself is *not* rotation invariant; ORB adds an orientation $\theta$ from the intensity centroid of a patch, and BRIEF sampling pairs are pre-rotated by $-\theta$. The resulting descriptor is identical for the same physical corner regardless of in-plane rotation.

**Scale invariance.** Plain ORB does *not* handle scale changes. OpenCV's implementation uses an image-pyramid (multiple downsampled levels) and detects corners at every scale, then propagates the level.

**Complexity.** Detection $O(\text{pixels} \cdot 16)$ for FAST checks. Descriptor extraction $O(N \cdot n_\text{bits})$ for $N$ keypoints. Hamming-distance matching $O(N_1 \cdot N_2 \cdot n_\text{bits})$ brute force — fast because $n_\text{bits}=256$ is a few CPU words.

**References.**
- Rublee, E., Rabaud, V., Konolige, K., & Bradski, G. (2011). *ORB: An efficient alternative to SIFT or SURF*. ICCV 2011.
- Rosten, E., & Drummond, T. (2006). *Machine learning for high-speed corner detection*. ECCV 2006. (FAST)
- Calonder, M., Lepetit, V., Strecha, C., & Fua, P. (2010). *BRIEF: Binary robust independent elementary features*. ECCV 2010.
